# 420 — Pool power & qualify contacts

Reads the windows from `410` and pools ERSP power inside each, then asks — per contact — whether
the response is **significant in the clustering sense, restricted to the window**.

**Pooling** = a **time-weighted average** of the contact's power over the window (boxcar = equal
weight inside the box; Gaussian = centre-weighted). Run for **two feature sets**, identically:
- **`hg`** — the single 70–150 Hz high-gamma line (the canonical iEEG task-response marker).
- **`bands15`** — the 15 frequency bands **separately**, on the native 300-bin time axis.

**Qualification** = clustering's high-activity gate (`prop(>2.2σ) ≥ 0.02` **OR**
`prop(<−3.0σ) ≥ 0.04`), computed **only over the window's time columns** — so it is *comparable*
to clustering but window-restricted. **Both signs are kept** (a contact can qualify as `+`
activation or `−` suppression). Gaussian gate support = centre ± 2σ (95% mass).

> ⚠️ *Comparable, not identical:* a contact that passes whole-ERSP clustering gating can fail a
> narrow window — that asymmetry is the confirmatory test, by design.

Optional secondary robustness: a per-contact **circular-time-shift null p** (`N_PERM > 0`).


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


In [ ]:
# ---------------- config knobs ----------------
cfg = P.load_window_config(P.OUTPUTS_ROOT / 'window_config.json')
USE_DS        = True                 # True: fast 15x30 ds grid · False: full-res 129x300
GRID          = 'ds' if USE_DS else 'full'
DS_TIME_BINS  = 30
FEATURE_SETS  = P.FEATURE_SETS       # ('hg', 'bands15')
WINDOW_SHAPES = P.WINDOW_SHAPES      # ('boxcar', 'gaussian')
N_PERM        = 0                    # >0 adds the circular-shift temporal-null p (slow)
SEED          = 42
print('grid:', GRID, '| zones:', list(cfg['zones']), '| feature sets:', FEATURE_SETS,
      '| shapes:', WINDOW_SHAPES, '| n_perm:', N_PERM)


In [ ]:
df_meta, X_full = P.prepare_pooling_dataset(INPUT_DIR)
X = P.downsample_dataset(X_full, time_bins=DS_TIME_BINS) if USE_DS else X_full
print('samples:', len(df_meta), '| grid:', GRID, '| X:', X.shape)


## Pool & gate (the heavy step — cached to `outputs/_dataset/pooling/pool_table_<grid>.parquet`)
One row per contact × condition × zone × window shape × feature. Re-run only when the windows
or the upstream ERSPs change.

> ⚠️ **ds caveat:** on the 15×30 grid the σ/proportion qualification gate runs on the *smoothed*
> band-mean map — a coarse proxy for the full-res clustering gate. Use `USE_DS=False` for the
> final qualification numbers; `ds` is for fast preliminary exploration.


In [ ]:
df_pool = P.build_pool_table(df_meta, X, cfg, grid=GRID,
                             feature_sets=FEATURE_SETS, window_shapes=WINDOW_SHAPES,
                             n_perm=N_PERM, seed=SEED)
print('pool table:', df_pool.shape, '| grid:', GRID)
df_pool.head()


## Qualifier summary
Distinct qualifying contacts per condition × zone × window shape × sign (the gate is
feature-independent, so features are collapsed first). Compare boxcar vs Gaussian counts —
close agreement = the zone is robust to the window choice.


In [ ]:
summ = P.qualifier_summary(df_pool)
display(summ)

import matplotlib.pyplot as plt
piv = summ.pivot_table(index=['condition', 'zone', 'sign'],
                       columns='window_shape', values='n_contacts', fill_value=0)
ax = piv.plot.barh(figsize=(9, 0.4 * len(piv) + 1))
ax.set_xlabel('# qualifying contacts'); ax.set_title('Qualifiers per zone (boxcar vs gaussian)')
plt.tight_layout(); plt.show()
